# CMPE 460 · HTTP Dataset Inspection

**Member 2 · Step 1**

This notebook summarizes the published training, validation, and test splits of the SecureAI-SE dataset. Run the cells from top to bottom. A CPU runtime in Google Colab is sufficient.

The tables summarize class counts, empty requests, request lengths in characters, exact duplicates, and requests with conflicting labels. Only aggregate statistics are displayed.

[Dataset card](https://huggingface.co/datasets/SecureAI-SE/http-attack-requests)


## 1. Load the Dataset

The `request` column contains the text the models will read, and `label` is the numeric ID of the ground-truth class. The class names come directly from the dataset metadata. The notebook prints the Hugging Face commit SHA used to load the data.


In [7]:
%pip -q install datasets


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import display

DATASET_ID = "SecureAI-SE/http-attack-requests"
SPLITS = ("train", "validation", "test")

revision = HfApi().dataset_info(DATASET_ID).sha
if not revision:
    raise RuntimeError("Could not determine the dataset revision.")

dataset = load_dataset(DATASET_ID, revision=revision)
label_names = dataset["train"].features["label"].names

frames = {
    split: dataset[split].to_pandas().loc[:, ["request", "label"]].copy()
    for split in SPLITS
}

print("Dataset revision:", revision)
display(pd.Series(label_names, name="class").rename_axis("label_id").to_frame())


Dataset revision: 9ada662402e0620cc4e3bd32b2cc3c2025bb3804


,class
label_id,
0,normal
1,sqli
2,xss
3,ssrf
4,ssti
5,lfi
6,traversal


## 2. Inspect Row Counts and Class Distributions

`p95_chars` is the 95th percentile of non-empty request lengths. About 95% of non-empty requests have at most this many characters. Use the training split to choose the maximum input sequence length in the preprocessing step.


In [9]:
def nonempty_requests(frame):
    text = frame["request"].astype("string")
    return frame.loc[text.str.strip().fillna("").ne("")]


summary_rows = []
for split, frame in frames.items():
    valid = nonempty_requests(frame)
    lengths = valid["request"].astype("string").str.len()

    summary_rows.append({
        "split": split,
        "rows": len(frame),
        "empty_requests": len(frame) - len(valid),
        "missing_labels": int(frame["label"].isna().sum()),
        "min_chars": lengths.min(),
        "median_chars": lengths.median(),
        "p95_chars": lengths.quantile(0.95),
        "max_chars": lengths.max(),
    })

summary = pd.DataFrame(summary_rows).set_index("split")
display(summary.round(1))

class_counts = pd.DataFrame({
    split: frame["label"].value_counts().reindex(
        range(len(label_names)), fill_value=0
    )
    for split, frame in frames.items()
})
class_counts.index = pd.Index(label_names, name="class")
display(class_counts)


,rows,empty_requests,missing_labels,min_chars,median_chars,p95_chars,max_chars
split,,,,,,,
train,4900,0,0,54,102.0,253.0,511
validation,700,0,0,55,103.0,248.0,541
test,1400,0,0,55,101.0,254.3,518


,train,validation,test
class,,,
normal,683,113,204
sqli,684,92,224
xss,706,100,194
ssrf,708,92,200
ssti,694,102,204
lfi,718,96,186
traversal,707,105,188


## 3. Check for Duplicates and Conflicting Labels

Two rows are exact duplicates when their request texts are identical. Within each split, every occurrence after the first counts as one duplicate row. Across splits, the table counts distinct request texts shared by each pair of splits. Empty requests are excluded from these counts.


In [10]:
from itertools import combinations

valid_frames = {
    split: nonempty_requests(frame) for split, frame in frames.items()
}

duplicate_rows = pd.Series({
    split: int(frame["request"].duplicated().sum())
    for split, frame in valid_frames.items()
}, name="duplicate_rows").rename_axis("split")
display(duplicate_rows.to_frame())

unique_requests = {
    split: set(frame["request"]) for split, frame in valid_frames.items()
}
overlaps = pd.DataFrame([
    {
        "split_a": first,
        "split_b": second,
        "shared_unique_requests": len(unique_requests[first] & unique_requests[second]),
    }
    for first, second in combinations(SPLITS, 2)
])
display(overlaps)

all_valid = pd.concat(valid_frames.values(), ignore_index=True)
labels_per_request = all_valid.groupby("request")["label"].nunique()
print("Distinct requests with conflicting labels:", int(labels_per_request.gt(1).sum()))


,duplicate_rows
split,
train,0
validation,0
test,0


,split_a,split_b,shared_unique_requests
0,train,validation,0
1,train,test,0
2,validation,test,0


Distinct requests with conflicting labels: 0


## 4. Initial Observations

Run the final cell to summarize the split sizes, class coverage, exact request overlap, and training request lengths. Use these observations to document your preprocessing decisions.


In [11]:
print("Split sizes and class coverage")
for split in SPLITS:
    row_count = int(summary.at[split, "rows"])
    counts = class_counts[split]
    present_classes = int(counts.gt(0).sum())
    print(
        f"- {split.capitalize()}: {row_count:,} rows, "
        f"{present_classes}/{len(label_names)} classes present "
        f"({counts.min():,}–{counts.max():,} rows per class)."
    )

print("\nExact duplicates and conflicting labels")
for overlap in overlaps.itertuples(index=False):
    print(
        f"- Shared distinct requests ({overlap.split_a} / {overlap.split_b}): "
        f"{overlap.shared_unique_requests:,}."
    )

conflict_count = int(labels_per_request.gt(1).sum())
print(f"- Distinct requests with conflicting labels: {conflict_count:,}.")

if overlaps["shared_unique_requests"].gt(0).any() or conflict_count:
    print("- Review the overlaps and label conflicts before choosing a preprocessing policy.")
else:
    print("- The exact-match audit found zero cross-split overlaps and zero label conflicts.")

print("\nTraining request lengths")
train_lengths = summary.loc["train", ["median_chars", "p95_chars"]]
if train_lengths.notna().all():
    print(
        f"- Median: {train_lengths['median_chars']:,.1f} characters. "
        f"95th percentile: {train_lengths['p95_chars']:,.1f} characters."
    )
else:
    print("- The training split contains no non-empty requests to measure.")


Split sizes and class coverage
- Train: 4,900 rows, 7/7 classes present (683–718 rows per class).
- Validation: 700 rows, 7/7 classes present (92–113 rows per class).
- Test: 1,400 rows, 7/7 classes present (186–224 rows per class).

Exact duplicates and conflicting labels
- Shared distinct requests (train / validation): 0.
- Shared distinct requests (train / test): 0.
- Shared distinct requests (validation / test): 0.
- Distinct requests with conflicting labels: 0.
- The exact-match audit found zero cross-split overlaps and zero label conflicts.

Training request lengths
- Median: 102.0 characters. 95th percentile: 253.0 characters.


In [ ]:
display(summary[["empty_requests", "missing_labels"]])
display(duplicate_rows.to_frame())

train_lengths = valid_frames["train"]["request"].str.len()
truncated = train_lengths.gt(256)
print(f"Training requests above 256 characters: {truncated.sum():,} ({truncated.mean():.2%})")

,empty_requests,missing_labels
split,,
train,0,0
validation,0,0
test,0,0


,duplicate_rows
split,
train,0
validation,0
test,0


Training requests above 256 characters: 227 (4.63%)


### Summary of Findings

All seven classes were represented in each split, with similar class counts. The exact-match audit found no shared requests across splits and no conflicting labels. In the training split, the median request length was 102 characters and the 95th percentile was 253 characters. Based on these statistics, we chose 256 characters as the initial maximum input length.